# Chapter 0 · Vision + Step 0 — your first Open Telco (OTel) model

_Where we're going, why, and the smallest first step that makes it real. (OTel = **Open Telco**, not OpenTelemetry.)_

## The north star
A telecom ops engineer faces a live incident — *"slow 5G in Riverside; several NR cells show low downlink throughput."* Instead of manually pulling KPIs, alarms, the right 3GPP clause, and business impact, they hand it to an **autonomous agent** that does all of it transparently and ends in a **human-gated** recommendation. That agent is [`../app/app.py`](../app/app.py) — a ReAct + reflection loop (Think → Act → Observe → Reflect) with OpenTelemetry-style span tracing. Two pieces start out fake, and this repo makes them real:

| Stand-in | What it becomes |
|---|---|
| `MockLLM` brain | a served **OTel LLM** |
| `retrieve_standards` keyword match | **OTel embedding + Vector Search** over a telco corpus |

## What Open Telco (OTel) is
A telecom-domain **RAG** stack (3GPP/GSMA/O-RAN) on HuggingFace under [`farbodtavakkoli`](https://huggingface.co/farbodtavakkoli): **embedding**, **reranker**, **LLM**, **safety/abstention**. On Databricks each maps to a serving primitive, governed in Unity Catalog and captured via inference tables — see [`../deploy/`](../deploy/) for the live build.

## Step 0 (this notebook)
Prove the models are real: load [`OTel-Embedding-335M`](https://huggingface.co/farbodtavakkoli/OTel-Embedding-335M) (BGE-large, ~335M, CPU-fine) and run **real semantic retrieval** over the same standards snippets the agent cites — swapping the keyword match for actual OTel embeddings.

**Next:** [`./01_otel_rag_pipeline.ipynb`](./01_otel_rag_pipeline.ipynb) assembles the full grounding pipeline.


## 1. Install dependencies

- **Databricks (serverless / standard cluster):** run the install cell below — it installs the deps and restarts Python. Only the *ML runtime* pre-ships `torch`/`transformers`; serverless does not.
- **Local (Jupyter):** run `pip install -r ../requirements.txt` in a terminal first, then skip the install cell.

In [0]:
# Local: uncomment to install. On Databricks the ML runtime already has torch + transformers.
%pip install -q "sentence-transformers>=3.0.0" "transformers>=4.40.0"              # local Jupyter — no restart needed

## 2. Load the model

First load downloads the weights from HuggingFace (a few hundred MB) and caches them. It runs on CPU by default — no `trust_remote_code` needed.

In [0]:
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_ID = "farbodtavakkoli/OTel-Embedding-335M"

model = SentenceTransformer(MODEL_ID)  # add device="cuda" later if a GPU is available
print(f"Loaded {MODEL_ID}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"Max sequence length : {model.max_seq_length}")

## 3. Sanity check — encode a couple of telecom sentences

The embedding is a fixed-length vector. Two sentences about the same concept should land close together (high cosine similarity); unrelated ones should not.

In [0]:
sentences = [
    "What is the F1 interface in O-RAN?",
    "The F1 interface connects the O-RAN Distributed Unit (O-DU) to the O-RAN Central Unit (O-CU).",
    "How do I bake sourdough bread at home?",
]

emb = model.encode(sentences, normalize_embeddings=True)
print("shape:", emb.shape)  # (3, 1024)

# With normalized vectors, cosine similarity is just the dot product.
print(f"F1-question  vs  F1-answer  : {emb[0] @ emb[1]:.3f}   (should be high)")
print(f"F1-question  vs  sourdough  : {emb[0] @ emb[2]:.3f}   (should be low)")

## 4. Real semantic retrieval over the `app/app.py` standards corpus

These are the same knowledge-base snippets the `app/app.py` ReAct agent cites in its `retrieve_standards` tool. In `app/app.py` that tool is a simple **keyword overlap** match. Here we replace it with **OTel embeddings** — actual semantic search.

In [0]:
# (citation, passage) — lifted from KNOWLEDGE_BASE in app/app.py
KNOWLEDGE_BASE = [
    ("3GPP TS 38.214 5.2",
     "Low SINR/CQI forces a lower MCS -> low per-UE throughput even at moderate PRB. "
     "Low throughput with LOW PRB load indicates a radio-quality (interference/coverage) problem, not congestion."),
    ("3GPP TS 36.213 7.2",
     "LTE downlink throughput saturates as PRB utilization approaches 100%. Sustained PRB >90% "
     "across neighboring cells in busy hour is the signature of CONGESTION (capacity limit), not radio quality."),
    ("O-RAN WG1 UC",
     "Congestion remediation order: (1) load-balance/traffic-steer to under-utilized neighbors, "
     "(2) enable/verify carrier aggregation, (3) add carrier/spectrum, (4) cell split / new site."),
    ("3GPP TS 36.331 8.1",
     "PCI collision between neighbor cells corrupts measurement reports and handovers, degrading SINR "
     "and raising drop/HO-failure rates. Resolve PCI conflicts before RF optimization."),
    ("RF Ops Playbook",
     "Correlate recurring EXTERNAL_INTERFERENCE_UL alarms with low-SINR cells before adjusting tilt or power."),
    ("TM Forum Open API",
     "Standardized management interfaces enable vendor-agnostic data collection across "
     "Huawei/Ericsson/Nokia OSS/BSS for closed-loop automation."),
]

cites   = [c for c, _ in KNOWLEDGE_BASE]
corpus  = [t for _, t in KNOWLEDGE_BASE]

# Encode the corpus once (in prod this becomes a Databricks Vector Search index).
corpus_emb = model.encode(corpus, normalize_embeddings=True)
print(f"Indexed {len(corpus)} standards passages -> {corpus_emb.shape}")

In [0]:
def retrieve_standards(query, k=3):
    """Semantic version of app/app.py's retrieve_standards tool, powered by OTel embeddings."""
    q = model.encode(query, normalize_embeddings=True)
    scores = corpus_emb @ q                    # cosine similarity (vectors are normalized)
    order = np.argsort(scores)[::-1][:k]
    return [(cites[i], float(scores[i]), corpus[i]) for i in order]


def show(query):
    print(f"\nQUERY: {query}")
    for cite, score, text in retrieve_standards(query):
        print(f"  [{score:.3f}] {cite:<20} {text[:90]}...")


# Two queries mirroring the two app/app.py scenarios: radio-quality vs. congestion.
show("cells show low downlink throughput but PRB load is low and SINR is poor")
show("every neighboring cell is at 98% PRB during the evening busy hour and users are slow")
show("two adjacent cells were configured with the same PCI")

Notice the model retrieves the *right* standard for each incident by **meaning**, not shared words — the low-throughput/low-load query surfaces the radio-quality clause (TS 38.214), the busy-hour query surfaces the congestion clause (TS 36.213), and the PCI query surfaces the PCI-collision clause (TS 36.331). That is exactly the grounding step the north-star agent needs.

## 5. What you just proved — and what's next

✅ The Open Telco models are real and load with two lines of code.  
✅ On CPU, `OTel-Embedding-335M` does genuine telecom semantic retrieval.  
✅ This is the first real component of the [`app/app.py`](../app/app.py) north-star loop — replacing its `retrieve_standards` keyword match with actual embeddings.

**Chapter 1** takes *this exact model* onto the platform:

1. Log it with `mlflow.sentence_transformers`.
2. Register it to **Unity Catalog** (governed + versioned).
3. Deploy a **Model Serving** endpoint.
4. Turn on the **inference table** so every request/response is captured.

That log → UC → serve → capture loop, proven once on the smallest model, is the template every other OTel model (reranker, LLM, safety) follows.

_See [`./00_starter_load_and_inference.ipynb`](./00_starter_load_and_inference.ipynb) for the full roadmap._